# MacroPulse — Regime-Filtered Momentum Strategy Demo

This notebook shows how to integrate the MacroPulse API into a live trading strategy.  
We build a **regime-filtered S&P 500 momentum strategy** that adjusts exposure based on the current macro regime.

**Key idea:** MacroPulse classifies the macro environment into four regimes:  
| Regime | Description | Default Equity Exposure |
|--------|-------------|-------------------------|
| `expansion` | Risk-on, Fed supporting growth | 100% |
| `recovery` | Healing from stress, liquidity re-injecting | 75% |
| `tightening` | Fed hiking, liquidity contracting | 25% |
| `risk_off` | Crisis / emergency mode | 0% |

By sizing down (or out) during macro stress, the strategy avoids the worst drawdowns while capturing most of the upside.

---
**Prerequisites**
```
pip install requests pandas numpy matplotlib yfinance
```
Set your API key in the cell below, or pass it as an environment variable `MACROPULSE_API_KEY`.

In [ ]:
import os
import datetime as dt
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
import yfinance as yf

# ── Configuration ────────────────────────────────────────────────────
API_BASE = os.getenv('MACROPULSE_BASE_URL', 'http://localhost:8000')
API_KEY  = os.getenv('MACROPULSE_API_KEY', '')   # paste your key here

HEADERS = {'X-MacroPulse-Key': API_KEY} if API_KEY else {}
print(f'MacroPulse base URL: {API_BASE}')
print(f'Auth: {"key set" if API_KEY else "dev-mode (no key required)"}')

## 1. Fetch the Latest Signal

In [ ]:
resp = requests.get(f'{API_BASE}/v1/signals/latest', headers=HEADERS)
resp.raise_for_status()
signal = resp.json()

regime = signal['regime']
liq    = signal['net_liquidity']
pca    = signal['pca_factors']
meta   = signal['model_metadata']

print('=== MacroPulse Signal Package ===')
print(f"Date            : {signal['date']}")
print(f"Macro Regime    : {regime['most_likely'].upper()}")
print(f"Confidence      : {regime['confidence']}")
print(f"Persistence     : {regime['persistence_days']} days")
print(f"Est. Remaining  : {regime['expected_duration_remaining_days']} days")
print(f"Risk Score      : {regime['risk_score']:.1f}")
print()
print('Regime Probabilities:')
for k, v in sorted(regime['probabilities'].items(), key=lambda x: -x[1]):
    bar = '█' * int(v * 30)
    print(f"  {k:<12} {bar:<30} {v:.1%}")
print()
print('Net Liquidity:')
print(f"  Level    : ${liq.get('level_bn', 'N/A')}B")
print(f"  4w Change: ${liq.get('change_4w_bn', 'N/A')}B")
print(f"  Z-Score  : {liq.get('zscore', 'N/A')}")
print(f"  Trend    : {liq.get('trend', 'N/A')}")
print()
print(f"Model Version  : {meta['model_version']}")

## 2. Fetch Historical Signal Range

In [ ]:
end_date   = dt.date.today()
start_date = end_date - dt.timedelta(days=730)  # ~2 years

resp = requests.get(
    f'{API_BASE}/v1/signals/range',
    params={'start': start_date.isoformat(), 'end': end_date.isoformat()},
    headers=HEADERS,
)
resp.raise_for_status()
history = resp.json()
print(f'Fetched {len(history)} trading days of signals ({start_date} → {end_date})')

## 3. Build a Regime DataFrame

In [ ]:
EXPOSURE = {
    'expansion' : 1.00,
    'recovery'  : 0.75,
    'tightening': 0.25,
    'risk_off'  : 0.00,
}

REGIME_COLORS = {
    'expansion' : '#2ecc71',
    'recovery'  : '#3498db',
    'tightening': '#e67e22',
    'risk_off'  : '#e74c3c',
}

rows = []
for s in history:
    r = s['regime']
    rows.append({
        'date'      : pd.Timestamp(s['date']),
        'regime'    : r['most_likely'],
        'confidence': r['confidence'],
        'risk_score': r['risk_score'],
        'prob_expansion' : r['probabilities'].get('expansion', 0),
        'prob_tightening': r['probabilities'].get('tightening', 0),
        'prob_risk_off'  : r['probabilities'].get('risk_off', 0),
        'prob_recovery'  : r['probabilities'].get('recovery', 0),
        'persistence_days': r['persistence_days'],
        'liq_zscore': s['net_liquidity'].get('zscore'),
        'liq_trend' : s['net_liquidity'].get('trend'),
    })

df = pd.DataFrame(rows).set_index('date').sort_index()
df['exposure'] = df['regime'].map(EXPOSURE)

print(df[['regime', 'confidence', 'risk_score', 'exposure']].tail(10).to_string())
print()
print('Regime distribution:')
print(df['regime'].value_counts().to_string())

## 4. Fetch SPX Returns and Build Equity Curves

In [ ]:
# Download SPX daily returns from Yahoo Finance
spx = yf.download('^GSPC', start=start_date, end=end_date + dt.timedelta(days=1),
                  progress=False, auto_adjust=True)
spx_ret = spx['Close'].pct_change().dropna()
spx_ret.index = pd.to_datetime(spx_ret.index).tz_localize(None)

# Align on common dates
common = df.index.intersection(spx_ret.index)
regime_aligned = df.loc[common]
ret_aligned    = spx_ret.loc[common]

print(f'Aligned {len(common)} trading days')

In [ ]:
def compute_equity_curve(returns: pd.Series, exposure: pd.Series) -> pd.Series:
    strategy_ret = returns * exposure
    return (1 + strategy_ret).cumprod()

def compute_sharpe(returns: pd.Series, exposure: pd.Series, rf: float = 0.0) -> float:
    r = returns * exposure
    ann = r.mean() * 252
    std = r.std() * np.sqrt(252)
    return (ann - rf) / std if std > 0 else 0.0

def compute_max_drawdown(equity_curve: pd.Series) -> float:
    rolling_max = equity_curve.cummax()
    dd = (equity_curve - rolling_max) / rolling_max
    return float(dd.min())

# Strategy: regime-filtered
strategy_curve = compute_equity_curve(ret_aligned, regime_aligned['exposure'])
bah_curve      = compute_equity_curve(ret_aligned, pd.Series(1.0, index=ret_aligned.index))

strategy_sharpe = compute_sharpe(ret_aligned, regime_aligned['exposure'])
bah_sharpe      = compute_sharpe(ret_aligned, pd.Series(1.0, index=ret_aligned.index))

strategy_mdd = compute_max_drawdown(strategy_curve)
bah_mdd      = compute_max_drawdown(bah_curve)

total_return_strategy = float(strategy_curve.iloc[-1]) - 1
total_return_bah      = float(bah_curve.iloc[-1]) - 1

print('=' * 50)
print(f'{"Metric":<25} {"Strategy":>10} {"Buy & Hold":>12}')
print('-' * 50)
print(f'{"Total Return":<25} {total_return_strategy:>10.1%} {total_return_bah:>12.1%}')
print(f'{"Sharpe Ratio":<25} {strategy_sharpe:>10.2f} {bah_sharpe:>12.2f}')
print(f'{"Max Drawdown":<25} {strategy_mdd:>10.1%} {bah_mdd:>12.1%}')
print('=' * 50)

## 5. Visualize: Regime Timeline + Equity Curves

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True,
                         gridspec_kw={'height_ratios': [1, 2, 1.5]})
fig.suptitle('MacroPulse Regime-Filtered Momentum Strategy', fontsize=15, fontweight='bold')

# ── Panel 1: Regime heatmap ────────────────────────────────────────
ax1 = axes[0]
ax1.set_ylabel('Regime', fontsize=10)
ax1.set_yticks([])

dates = regime_aligned.index
prev_regime = None
block_start = dates[0]
for i, (date, row) in enumerate(regime_aligned.iterrows()):
    cur = row['regime']
    if cur != prev_regime and prev_regime is not None:
        ax1.axvspan(block_start, date,
                    color=REGIME_COLORS.get(prev_regime, '#aaa'), alpha=0.6)
        block_start = date
    prev_regime = cur
ax1.axvspan(block_start, dates[-1],
            color=REGIME_COLORS.get(prev_regime, '#aaa'), alpha=0.6)

# Legend
from matplotlib.patches import Patch
ax1.legend(
    [Patch(facecolor=c, alpha=0.6, label=l) for l, c in REGIME_COLORS.items()],
    REGIME_COLORS.keys(),
    loc='center right', ncol=4, fontsize=8,
)

# ── Panel 2: Equity curves ────────────────────────────────────────
ax2 = axes[1]
ax2.plot(strategy_curve.index, strategy_curve.values,
         color='#2ecc71', linewidth=2, label='MacroPulse Strategy')
ax2.fill_between(strategy_curve.index, 1, strategy_curve.values,
                 color='#2ecc71', alpha=0.15)
ax2.plot(bah_curve.index, bah_curve.values,
         color='#aaa', linewidth=1.5, linestyle='--', label='Buy & Hold')
ax2.axhline(1.0, color='black', linewidth=0.5, linestyle=':')
ax2.set_ylabel('Cumulative Return', fontsize=10)
ax2.legend(fontsize=9)

# Stats box
stats_text = (
    f"Strategy  Sharpe: {strategy_sharpe:.2f}   MaxDD: {strategy_mdd:.1%}\n"
    f"Buy&Hold  Sharpe: {bah_sharpe:.2f}   MaxDD: {bah_mdd:.1%}"
)
ax2.text(0.02, 0.97, stats_text,
         transform=ax2.transAxes, fontsize=8, verticalalignment='top',
         bbox=dict(boxstyle='round,pad=0.4', facecolor='white', alpha=0.8))

# ── Panel 3: Regime exposure ──────────────────────────────────────
ax3 = axes[2]
ax3.fill_between(regime_aligned.index, 0, regime_aligned['exposure'],
                 step='post', alpha=0.5, color='#3498db', label='Equity Exposure')
ax3.set_ylabel('Exposure', fontsize=10)
ax3.set_ylim(-0.05, 1.15)
ax3.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax3.set_yticklabels(['0%', '25%', '50%', '75%', '100%'])
ax3.legend(fontsize=9)

# Date formatting
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax3.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=30, ha='right')

plt.tight_layout()
plt.savefig('macropulse_regime_strategy.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: macropulse_regime_strategy.png')

## 6. Regime Probability Heatmap

In [ ]:
prob_cols = ['prob_expansion', 'prob_recovery', 'prob_tightening', 'prob_risk_off']
prob_df = regime_aligned[prob_cols].resample('W').mean()  # weekly averages

fig, ax = plt.subplots(figsize=(14, 3))
im = ax.imshow(
    prob_df.T.values,
    aspect='auto',
    cmap='RdYlGn',
    vmin=0, vmax=1,
    extent=[mdates.date2num(prob_df.index[0]), mdates.date2num(prob_df.index[-1]), -0.5, 3.5]
)
ax.set_yticks(range(4))
ax.set_yticklabels(['risk_off', 'tightening', 'recovery', 'expansion'])
ax.xaxis_date()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=30, ha='right')
plt.colorbar(im, ax=ax, label='Probability')
ax.set_title('Weekly Average Regime Probabilities', fontsize=12)
plt.tight_layout()
plt.show()

## 7. Live Integration Pattern

Here's how to integrate MacroPulse into your own strategy loop:

In [ ]:
def get_current_exposure(
    api_base: str,
    api_key: str,
    custom_exposure: dict | None = None,
) -> dict:
    """
    Fetch the latest MacroPulse signal and return a trading decision.

    Returns:
        {
          'regime': str,
          'confidence': str,
          'exposure': float,          # 0.0 – 1.0
          'persistence_days': int,
          'liq_trend': str,
          'raw': dict,                # full signal package
        }
    """
    exposure_map = custom_exposure or {
        'expansion' : 1.00,
        'recovery'  : 0.75,
        'tightening': 0.25,
        'risk_off'  : 0.00,
    }

    headers = {'X-MacroPulse-Key': api_key} if api_key else {}
    resp = requests.get(f'{api_base}/v1/signals/latest', headers=headers, timeout=10)
    resp.raise_for_status()
    pkg = resp.json()

    regime = pkg['regime']['most_likely']
    confidence = pkg['regime']['confidence']
    persistence = pkg['regime']['persistence_days']
    liq_trend = pkg['net_liquidity']['trend']
    base_exposure = exposure_map.get(regime, 0.5)

    # Optional: reduce exposure further if LOW confidence or liquidity contracting
    if confidence == 'LOW':
        base_exposure *= 0.8
    if liq_trend == 'CONTRACTING' and regime in ('expansion', 'recovery'):
        base_exposure *= 0.9

    return {
        'regime': regime,
        'confidence': confidence,
        'exposure': round(base_exposure, 4),
        'persistence_days': persistence,
        'liq_trend': liq_trend,
        'raw': pkg,
    }


# Example usage
decision = get_current_exposure(API_BASE, API_KEY)
print(f"Today's Decision")
print(f"  Regime      : {decision['regime']} ({decision['confidence']})")
print(f"  Persistence : {decision['persistence_days']} days")
print(f"  Liq Trend   : {decision['liq_trend']}")
print(f"  → Equity Exposure: {decision['exposure']:.0%}")

---

## Next Steps

- **Get your API key**: [macropulse.io](https://macropulse.io) — Free tier includes 50 requests/day
- **View the live dashboard**: `GET /dashboard` — interactive Plotly charts, no key required
- **Backtest with history**: `GET /v1/signals/range?start=2022-01-01&end=2024-12-31`
- **Performance attribution**: `GET /v1/performance?lookback=252`
- **Forecasts**: `GET /v1/forecast/regime?horizon=5` — 5-day ahead regime probability forecast

### API Reference
| Endpoint | Description |
|----------|-------------|
| `GET /v1/signals/latest` | Latest unified signal package |
| `GET /v1/signals/{date}` | Historical signal by date |
| `GET /v1/signals/range` | Signal time series |
| `GET /v1/regime/current` | Current regime + probabilities |
| `GET /v1/performance` | Regime performance attribution |
| `GET /v1/forecast/regime` | 5-day ahead ARIMA forecast |
| `GET /v1/liquidity/history` | Net liquidity time series |
| `GET /dashboard` | Interactive dashboard (no auth) |

> Built with ❤️ by MacroPulse — macro intelligence for systematic traders.